# COVID-19 Chest X-ray Classifier — Deploy, Test and Serve

This notebook takes the model registered by `01_train_model.ipynb` and:

1. deploys it as a **managed online endpoint** (a REST API),
2. tests it on images from every class,
3. predicts **any new X-ray** you upload, and
4. launches the **X-Ray Insight** web app.

The notebook is safe to *Run all*: it reuses one fixed endpoint name and the clean-up cell is disabled.

⚠️ An endpoint is billed for every hour it exists. Delete it when your demo is finished (last cell).

In [ ]:
%pip install -q azure-ai-ml azure-identity requests

In [ ]:
import os, json, base64
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (ManagedOnlineEndpoint, ManagedOnlineDeployment,
                                  Environment, CodeConfiguration)
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Workspace:", ml_client.workspace_name)

ENDPOINT_NAME = "covid-xray-aditi"     # one fixed name: re-running never creates extra endpoints
TEST_DIR = "Covid19-dataset/test"
model = ml_client.models.get("covid-xray-resnet50", label="latest")
print(f"Using model {model.name} v{model.version}")

## 1. Create (or reuse) the endpoint

In [ ]:
existing = [e.name for e in ml_client.online_endpoints.list()]
if ENDPOINT_NAME not in existing:
    ml_client.online_endpoints.begin_create_or_update(ManagedOnlineEndpoint(
        name=ENDPOINT_NAME, auth_mode="key",
        description="COVID-19 / Normal / Viral Pneumonia chest X-ray classifier")).result()
    print("Endpoint created:", ENDPOINT_NAME)
else:
    print("Endpoint already exists, reusing it:", ENDPOINT_NAME)

## 2. Deploy the model

- **Environment:** Ubuntu base image + `src/conda.yml` (CPU PyTorch + Azure ML inference server).
- **Code:** `src/score.py` — `init()` loads the model once, `run()` predicts each request.
- **VM size:** `Standard_DS1_v2`. The Azure for Students subscription has a 6-vCPU quota and Azure reserves
  double capacity during deployment, so a 1-vCPU VM is what fits next to the compute instance.

The first build takes 10–20 minutes (Docker image).

In [ ]:
env = Environment(name="covid-xray-inference-env", conda_file="src/conda.yml",
                  image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest")

deployment = ManagedOnlineDeployment(
    name="blue", endpoint_name=ENDPOINT_NAME, model=model, environment=env,
    code_configuration=CodeConfiguration(code="src", scoring_script="score.py"),
    instance_type="Standard_DS1_v2", instance_count=1)
ml_client.online_deployments.begin_create_or_update(deployment).result()

ep = ml_client.online_endpoints.get(ENDPOINT_NAME)
ep.traffic = {"blue": 100}                      # new deployments get 0% traffic by default
ml_client.online_endpoints.begin_create_or_update(ep).result()
print("Deployment ready:", ml_client.online_endpoints.get(ENDPOINT_NAME).traffic)

In [ ]:
# If the deployment fails, the container logs usually explain why (e.g. an error in score.py)
print(ml_client.online_deployments.get_logs(name="blue", endpoint_name=ENDPOINT_NAME, lines=30))

## 3. Test the endpoint (2 images per class)

In [ ]:
def to_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

paths = [os.path.join(TEST_DIR, c, f) for c in sorted(os.listdir(TEST_DIR))
         for f in sorted(os.listdir(os.path.join(TEST_DIR, c)))[:2]]
with open("batch_request.json", "w") as f:
    json.dump({"images": [to_b64(p) for p in paths]}, f)
with open("sample_request.json", "w") as f:       # paste this into the Studio "Test" tab
    json.dump({"image": to_b64(paths[0])}, f)

results = json.loads(ml_client.online_endpoints.invoke(
    endpoint_name=ENDPOINT_NAME, request_file="batch_request.json"))
correct = 0
for p, r in zip(paths, results):
    actual = p.split("/")[-2]
    correct += actual == r["prediction"]
    print(f"actual: {actual:16s} predicted: {r['prediction']:16s} confidence: {r['confidence']:.2f}")
print(f"\n{correct}/{len(paths)} correct on endpoint {ENDPOINT_NAME}")

## 4. Call the endpoint over plain HTTP (how any app uses it)

In [ ]:
import requests
scoring_uri = ml_client.online_endpoints.get(ENDPOINT_NAME).scoring_uri
key = ml_client.online_endpoints.get_keys(ENDPOINT_NAME).primary_key

r = requests.post(scoring_uri, json={"image": to_b64(paths[0])},
                  headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"})
print("Scoring URI:", scoring_uri)
print(r.status_code, r.json())

## 5. Predict a NEW X-ray image

Upload any chest X-ray (JPG/PNG) into this folder and put its file name in `NEW_IMAGE`.

In [ ]:
def predict_new_image(image_path, endpoint_name=ENDPOINT_NAME):
    """Send any new X-ray image to the deployed model and return its prediction."""
    with open("new_request.json", "w") as f:
        json.dump({"image": to_b64(image_path)}, f)
    result = json.loads(ml_client.online_endpoints.invoke(
        endpoint_name=endpoint_name, request_file="new_request.json"))[0]
    print(f"Image: {image_path}")
    print(f"Prediction: {result['prediction']}  (confidence {result['confidence']:.0%})")
    for cls, p in result["probabilities"].items():
        print(f"   {cls:16s} {p:.1%}")
    if result["confidence"] < 0.6:
        print("   Low confidence - the model is unsure about this image.")
    return result

NEW_IMAGE = "images.jpeg"          # just the file name: the notebook runs inside this folder
if os.path.exists(NEW_IMAGE):
    predict_new_image(NEW_IMAGE)
else:
    print("File not found:", NEW_IMAGE, "- upload it into this folder and check the exact name.")

## 6. Launch the X-Ray Insight web app

Runs `app/app.py` with Streamlit on the compute instance. Open
`https://<compute-instance-name>-8501.<region>.instances.azureml.ms` (you must be signed in to Azure).

In [ ]:
%pip install -q streamlit
import subprocess, sys, time

subprocess.run(["pkill", "-f", "streamlit run"])     # stop an older copy if still running
time.sleep(2)
app = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app/app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--theme.base", "light", "--theme.primaryColor", "#0B3D91",
     "--browser.gatherUsageStats", "false"],
    stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(10)
print(open("streamlit.log").read()[-400:])
print("App started. Open: https://ci-covid-8501.eastasia.instances.azureml.ms  (replace with your compute name/region)")

## 7. Clean up (stops billing)

Disabled on purpose so *Run all* can never delete the endpoint. When you are done, remove the `#` and run **only** this cell.
The registered model stays in the workspace, so you can redeploy later.

In [ ]:
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print("Endpoint deleted")
print("Clean-up is disabled - endpoint", ENDPOINT_NAME, "is still running. Delete it after your demo.")